In [1]:
import pybedtools
from pybedtools import BedTool

load specific files

In [10]:
genome = 'GRCh38_EBV.chrom.sizes.tsv'

hot_loci_name = 'hot_regions.clean.bed'
h3k4me3_name = 'K562_H3K4me3.bed'
h3k27me3_name = 'K562_H3k27me3.bed'

hot = BedTool(hot_loci_name).sort(g=genome)
active = BedTool(h3k4me3_name).sort(g=genome)
inactive = BedTool(h3k27me3_name).sort(g=genome)

n_hot = hot.count()

print(hot.count(), 'hot regions')
print(active.count(), 'active regions')
print(inactive.count(), 'inactive regions')

5990 hot regions
21445 active regions
163784 inactive regions


In [3]:
def hit_set(hot_bt, mark_bt):
    return {f"{iv.chrom}:{iv.start}-{iv.end}" for iv in hot_bt.intersect(mark_bt, u=True)}
 
hits_active = hit_set(hot, active)
hits_inactive = hit_set(hot, inactive)

print('overlap between regions of active/inactive promoters and HOT regions')
print(f'active: {len(hits_active)}/{n_hot} loci overlap')
print(f'inactive: {len(hits_inactive)}/{n_hot} loci overlap')

overlap between regions of active/inactive promoters and HOT regions
active: 860/5990 loci overlap
inactive: 118/5990 loci overlap


but what is this actually telling me ???

In [11]:
intervals = {}
kind = {}

for locus in hot:
    
    interval_id = f'{locus.chrom}:{locus.start}-{locus.end}'
    intervals[interval_id] = locus

    is_active = interval_id in hits_active
    is_inactive = interval_id in hits_inactive

    if is_active and is_inactive:
        kind[interval_id] = 'both'
    elif is_active:
        kind[interval_id] = 'active_only'
    elif is_inactive:
        kind[interval_id] = 'inactive_only'
    else:
        kind[interval_id] = 'neither'

In [5]:
from collections import Counter
counts = Counter(kind.values())

print('of all HOT loci')
for category in ['both', 'active_only', 'inactive_only', 'neither']:
    n = counts.get(category, 0)
    percent = 100 * n / n_hot if n_hot else 0
    print(f'{category}: {n} ({percent:.2f}%)')

of all HOT loci
both: 15 (0.25%)
active_only: 845 (14.11%)
inactive_only: 103 (1.72%)
neither: 5027 (83.92%)


In [6]:
both = active.intersect(inactive).sort(g=genome).merge()
n_both = both.count()
print(n_both)

3692


In [7]:
both_in_hot = both.intersect(hot, u=True)
both_not_in_hot = both.intersect(hot, v=True)

n_hot_overlap = both_in_hot.count()
percent_hot = 100 * n_hot_overlap / n_both

n_not_hot_overlap = both_not_in_hot.count()
percent_not_hot = 100 * n_not_hot_overlap / n_both

print('of all regions which have both an active (H3K4me3) and inactive (H3k27me3) mark')
print(f'hot region    : {n_hot_overlap:5d} {percent_hot:.2f}%')
print(f'not hot region: {n_not_hot_overlap:5d} {percent_not_hot:.2f}%')

of all regions which have both an active (H3K4me3) and inactive (H3k27me3) mark
hot region    :     9 0.24%
not hot region:  3683 99.76%


run fisher test to check if hot regions are enriched among samples with an active an inactive mark

In [8]:
out = both.fisher(hot, g = genome)
print(out)

# Number of query intervals: 3692
# Number of db intervals: 5990
# Number of overlaps: 9
# Number of possible intervals (estimated): 594407
# phyper(9 - 1, 3692, 594407 - 3692, 5990, lower.tail=F)
# Contingency Table Of Counts
#_________________________________________
#           |  in -b       | not in -b    |
#     in -a | 9            | 3683         |
# not in -a | 5981         | 584734       |
#_________________________________________
# p-values for fisher's exact test
left	right	two-tail	ratio
2.859e-08	1	5.3274e-08	0.239

